In [0]:
import pandas as pd                                                     # Import pandas for data cleaning
import seaborn as sns                                                   # Import Seaborn for visualization
import matplotlib.pyplot as plt                                         # Import matplotlib library for visualization
import plotly.express as px                                             # Import plotly library for visualization
import plotly.graph_objects as go

from pyspark.sql import functions as F
from pyspark.sql.functions import col                                   # TableFunctions
from pyspark.sql.functions import mean, min, max, stddev,count          # MathsFunctions
from pyspark.sql.functions import to_date,month,year,datediff           # DateFunctions
from pyspark.sql.functions import abs                                   # OtherFunctions

In [0]:
# %sql
# -- DATA INTEGRATION & SCHEMA ENFORCEMENT
# CREATE OR REPLACE TABLE samplesuperstore.bronzedata.orders
# PARTITIONED BY (order_date, region)
# AS
# SELECT * from samplesuperstore.bronzedata.orders_main
#     TRY_CAST(row_id AS INT)            AS row_id,
#     TRY_CAST(order_id AS STRING)       AS order_id,
#     TRY_CAST(order_date AS DATE)       AS order_date,
#     TRY_CAST(ship_date AS DATE)        AS ship_date,
#     TRY_CAST(ship_mode AS STRING)      AS ship_mode,
#     TRY_CAST(customer_id AS STRING)    AS customer_id,
#     TRY_CAST(customer_name AS STRING)  AS customer_name,
#     TRY_CAST(segment AS STRING)        AS segment,
#     TRY_CAST(country AS STRING)        AS country,
#     TRY_CAST(city AS STRING)           AS city,
#     TRY_CAST(state AS STRING)          AS state,
#     TRY_CAST(postal_code AS INT)       AS postal_code,
#     TRY_CAST(region AS STRING)         AS region,
#     TRY_CAST(product_id AS STRING)     AS product_id,
#     TRY_CAST(category AS STRING)       AS category,
#     TRY_CAST(sub_category AS STRING)   AS sub_category,
#     TRY_CAST(product_name AS STRING)   AS product_name,
#     TRY_CAST(sales AS DOUBLE)          AS sales, 
#     TRY_CAST(quantity AS INT)          AS quantity,
#     TRY_CAST(discount AS DOUBLE)       AS discount,
#     TRY_CAST(profit AS DOUBLE)         AS profit,
#     TRY_CAST(price AS DOUBLE)          AS price,
#     TRY_CAST(cost AS DOUBLE)           AS cost
# FROM samplesuperstore.bronzedata.orders_main;

# -- Compact files and apply ZORDER on common filter columns
# OPTIMIZE samplesuperstore.silver.orders
# ZORDER BY (customer_id, order_date);



In [0]:
# Notebook 1: Financial Silver
orders = spark.read.table("samplesuperstore.bronzedata.orders")

silver_financial = orders.select("Sales","Profit","Discount","Order_ID")
silver_financial.write.mode("overwrite").saveAsTable("samplesuperstore.silverdata.orders_financial")

# Notebook 2: Shipping Silver
silver_shipping = orders.select("Ship_Mode","Ship_Date","Order_ID")
silver_shipping.write.mode("overwrite").saveAsTable("samplesuperstore.silverdata.orders_shipping")


In [0]:
from pyspark.sql import SparkSession, functions as F


# 🔹 Load the latest schema directly (no REFRESH TABLE)
df = orders

# 🔹 Function to compute stats for one column
def column_stats(df, colname):
    # Compute quantiles in one call
    quantiles = df.approxQuantile(colname, [0.05, 0.25, 0.5, 0.75, 0.95], 0.01)
    p5, q1, median, q3, p95 = quantiles
    iqr = q3 - q1

    lower_outlier = q1 - 1.5 * iqr
    upper_outlier = q3 + 1.5 * iqr

    # Collect stats in one select
    stats = df.select(
        F.round(F.mean(colname), 2).alias("Mean"),
        F.lit(round(median, 2)).alias("Median"),
        F.round(F.min(colname), 2).alias("Min"),
        F.round(F.max(colname), 2).alias("Max"),
        F.round(F.stddev(colname), 2).alias("StdDev"),
        F.count(F.when(F.col(colname).isNull(), 1)).alias("Nulls"),
        F.lit(round(lower_outlier, 2)).alias("Lower_Outlier"),
        F.lit(round(upper_outlier, 2)).alias("Upper_Outlier"),
        F.count(F.when((F.col(colname) < lower_outlier) | (F.col(colname) > upper_outlier), 1)).alias("Total_Outlier_Count"),
        F.lit(round(p5, 2)).alias("Lower_Anomaly"),
        F.lit(round(p95, 2)).alias("Upper_Anomaly"),
        F.count(F.when((F.col(colname) < p5) | (F.col(colname) > p95), 1)).alias("Total_Anomaly_Count")
    ).first()

    return stats

# 🔹 Detect numeric columns automatically
numeric_types = {"long", "double", "integer"}
numeric_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() in numeric_types]

# 🔹 Loop through numeric columns
summary_data = []
for col in numeric_cols:
    stats = column_stats(df, col)
    summary_data.append((col,) + tuple(stats))

# 🔹 Build summary DataFrame
summary_df = spark.createDataFrame(
    summary_data,
    ["Fact", "Mean", "Median", "Min", "Max", "StdDev", "Nulls",
     "Lower_Outlier", "Upper_Outlier", "Total_Outlier_Count",
     "Lower_Anomaly", "Upper_Anomaly", "Total_Anomaly_Count"]
)

# 🔹 Display and save
display(summary_df)

summary_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("samplesuperstore.silverdata.orders_summary")


In [0]:
#HANDLING NULLS
def nulls_summary(table_name):
    df = orders
    total_records = df.count()
    summary = []
    for col in df.columns:
        nulls = df.filter(F.col(col).isNull()).count()
        not_nulls = total_records - nulls
        summary.append((table_name, col, total_records, not_nulls, nulls))
    return summary

    # Collect summaries for all three tables
orders_summary  = nulls_summary("samplesuperstore.bronzedata.orders")
people_summary  = nulls_summary("samplesuperstore.bronzedata.people")
returns_summary = nulls_summary("samplesuperstore.bronzedata.returns")

    # Combine into one DataFrame
summary_df = spark.createDataFrame(
    orders_summary + people_summary + returns_summary,
    ["Table", "Column", "TotalRecords", "NotNulls", "Nulls"]
)
summary_df.write.mode("overwrite").saveAsTable("samplesuperstore.silverdata.eda_handlingnulls")

In [0]:
from pyspark.sql import functions as F

# Step 1: Compute stats safely (ignore NULLs automatically)
stats = orders.select(
    F.mean("sales").alias("mean_sales"),
    F.stddev("sales").alias("std_sales"),
    F.mean("profit").alias("mean_profit"),
    F.stddev("profit").alias("std_profit")
).collect()[0]

mean_sales, std_sales = stats["mean_sales"], stats["std_sales"]
mean_profit, std_profit = stats["mean_profit"], stats["std_profit"]

# Step 2: Flag outliers (handle NULLs explicitly)
orders_outliers = orders.withColumn(
    "SalesOutlier",
    F.when(
        (F.col("sales").isNotNull()) &
        ((F.col("sales") > mean_sales + 3*std_sales) | (F.col("sales") < mean_sales - 3*std_sales)),
        1
    ).otherwise(0)
).withColumn(
    "ProfitOutlier",
    F.when(
        (F.col("profit").isNotNull()) &
        ((F.col("profit") > mean_profit + 3*std_profit) | (F.col("profit") < mean_profit - 3*std_profit)),
        1
    ).otherwise(0)
)

# Step 3: Save with schema overwrite
orders_outliers.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("samplesuperstore.silverdata.orders_outliers")


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from sklearn.ensemble import IsolationForest

# 🔹 Select numeric columns for anomaly detection
numeric_cols = [
    "sales", "profit", "quantity", "discount",
    "row_id"
]

# Assemble features into a single vector
assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
orders_vec = assembler.transform(orders)

# Convert to Pandas for scikit-learn
orders_pd = orders_vec.select(numeric_cols).toPandas()

# 🔹 Run Isolation Forest
iforest = IsolationForest(
    contamination=0.05,        # expected % anomalies
    random_state=42
)
orders_pd["anomaly_pred"] = iforest.fit_predict(orders_pd[numeric_cols])
orders_pd["anomaly_score"] = iforest.decision_function(orders_pd[numeric_cols])

# Convert back to Spark DataFrame
orders_anomaly = spark.createDataFrame(orders_pd)

# 🔹 Save results back to Silver layer
orders_anomaly.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("samplesuperstore.silverdata.orders_anomaly")


In [0]:
# # Cell 1: Install D-Tale (run once in VS Code terminal)
# pip install dtale

# # Cell 2: Import libraries
# import pandas as pd
# import dtale

# # Cell 3: Load your Sample Super Store dataset
# df = pd.read_csv(r"C:\Users\dhpur\Desktop\Orders.csv")

# # Cell 4: Convert Order Date & Ship Date into datetime format
# df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
# df['Ship Date'] = pd.to_datetime(df['Ship Date'], errors='coerce')

# # Optional: set Order Date as index if you want time-series exploration
# # df = df.set_index('Order Date')

# # Cell 5: Launch D-Tale
# d = dtale.show(df)
# d.open_browser()
